# Riemann Hypothesis — Spectral Sketch

This notebook illustrates the heuristic connection between the Riemann zeta
zeros and GUE eigenvalue statistics, using `spectral_sandbox`.

> **Disclaimer:** This is an exploratory / educational notebook.  It does not
> constitute a proof of the Riemann Hypothesis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectral_sandbox import (
    random_hermitian,
    riemann_zeta_operator,
    isep_check,
    ProofVault,
)
from spectral_sandbox.proofvault import ClaimStatus

## 1. Approximate Riemann Zeta Zeros

We use the asymptotic Berry–Keating formula to estimate the imaginary parts
of the first *n* non-trivial zeros.

In [ ]:
n = 100
Z = riemann_zeta_operator(n)
zeta_zeros = np.diag(Z)

print("First 10 approximate zeta zero heights:")
print(np.round(zeta_zeros[:10], 4))

## 2. ISEP Check on the Zeta Operator

In [ ]:
passes, error = isep_check(Z)
print(f"ISEP satisfied: {passes}  (residual {error:.2e})")

## 3. Level-Spacing: Zeta Zeros vs GUE

Montgomery's pair-correlation conjecture (1973) predicts that the normalised
nearest-neighbour spacings of zeta zeros follow the GUE distribution.

In [ ]:
# Normalised spacings of zeta zeros
sp_zeta = np.diff(zeta_zeros)
sp_zeta /= sp_zeta.mean()

# Normalised spacings of a GUE sample of the same size
H = random_hermitian(n, seed=0)
eigs_gue = np.linalg.eigvalsh(H)
sp_gue = np.diff(np.sort(eigs_gue))
sp_gue /= sp_gue.mean()

# Wigner surmise
s = np.linspace(0, 4, 300)
wigner = (np.pi / 2) * s * np.exp(-np.pi * s**2 / 4)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, sp, title in zip(axes, [sp_zeta, sp_gue],
                          ['Approximate Zeta Zeros', 'GUE sample']):
    ax.hist(sp, bins=30, density=True, alpha=0.65, label='Data')
    ax.plot(s, wigner, 'r-', lw=2, label='Wigner surmise')
    ax.set_title(title)
    ax.set_xlabel('Normalised spacing s')
    ax.legend()
axes[0].set_ylabel('p(s)')
plt.suptitle('Level-Spacing Distributions (n = 100)', y=1.01)
plt.tight_layout()
plt.show()

## 4. ProofVault: Tracking Our Claims

In [ ]:
vault = ProofVault()

vault.register(
    "rh_open",
    "All non-trivial zeros of ζ(s) lie on Re(s) = 1/2.",
    status=ClaimStatus.OPEN,
)
vault.register(
    "isep_zeta_approx",
    f"The approximate {n}×{n} zeta operator satisfies ISEP.",
)
if passes:
    vault.verify("isep_zeta_approx",
                 description="Checked numerically",
                 data={"n": n, "error": float(error)})

vault.register(
    "gue_montgomery",
    "Zeta-zero spacings follow the GUE pair-correlation (Montgomery's conjecture).",
    status=ClaimStatus.CONJECTURE,
)

print(vault.to_json())